# 📊 Polymarket: Datafication & the Wisdom(?) of Crowds


In [11]:
# install dependencies, import what we need

!pip install requests pandas websocket-client --quiet


[notice] A new release of pip is available: 23.2.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [12]:
import requests
import pandas as pd
import json

print('✅ libraries loaded')

✅ libraries loaded


---
## 1️⃣ First - what is a 'market' in this case?

We'll use the **Gamma API** — Polymarket's free public API.

Check out their 'Getting Started' documentation: https://help.polymarket.com/en/articles/13364060-what-is-polymarket

Take some notes here:

- What is a 'market,' in Polymarket's world? 
- Are there things that you have questions about, from this documentation alone?

In [3]:
# query the API

GAMMA = "https://gamma-api.polymarket.com"

# make a request here
resp = requests.get(f"{GAMMA}/markets", params={
    "active": "true",
    "closed": "false",
    "limit": 30,
    "order": "volume24hr",
    "ascending": "false"
})

# store the JSON object in markets_raw

markets_raw = resp.json()
print(f"✅ Fetched {len(markets_raw)} markets")

✅ Fetched 30 markets


---
## DataFrame

Each row = one 'market'. 

According to Polymarket, prices are **implied probabilities** (0 to 1). What do you think about this connection?

In [13]:
# print your results from the API

print(markets_raw)

[{'id': '2036399', 'question': 'US x Iran ceasefire extended by April 22, 2026?', 'conditionId': '0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cdc09e9d47e7d54356c6cd', 'slug': 'us-x-iran-ceasefire-extended-by-april-22-2026', 'resolutionSource': '', 'endDate': '2026-04-21T00:00:00Z', 'liquidity': '6655208.72', 'startDate': '2026-04-20T19:37:22.15054Z', 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/us-x-iran-ceasefire-by-Cgmx3GCuOwjs.jpg', 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/us-x-iran-ceasefire-by-Cgmx3GCuOwjs.jpg', 'description': 'This market will resolve to “Yes” if there is an official extension of the two-week ceasefire agreement between the United States and Iran announced on April 7, 2026, defined as a publicly announced and mutually agreed extension to the halt in direct military engagement between the United States and Iran, by the specified date, 11:59 PM ET. Otherwise, this market will resolve to "No".\n\nBoth extensions of the April 7 

In [14]:
# use this cell to put the results in a pandas dataFrame
















In [5]:
# use this cell to find the top 10 'market' questions by trading VOLUME







,Market Question,YES Probability,24h Volume ($),Closes
0,"US x Iran ceasefire extended by April 22, 2026?",0.35%,1713175953.22%,2026-04-21
1,Will there be no change in Fed interest rates after the April 2026 meeting?,99.95%,1095368733.88%,2026-04-29
2,Will the Fed increase interest rates by 25+ bps after the April 2026 meeting?,0.05%,1063846625.36%,2026-04-29
3,Will the Fed decrease interest rates by 25 bps after the April 2026 meeting?,0.05%,984953537.03%,2026-04-29
4,Will Arsenal FC win on 2026-04-29?,15.50%,869530687.15%,2026-04-29
5,"Will Bitcoin hit $150k by June 30, 2026?",1.35%,582165289.42%,2026-07-01
6,Will the Iranian regime fall by April 30?,0.05%,424869721.97%,2026-04-30
7,Will Club Atlético de Madrid win on 2026-04-29?,12.00%,355818457.52%,2026-04-29
8,Indian Premier League: Mumbai Indians vs Sunrisers Hyderabad,0.05%,271520008.58%,2026-05-06
9,Will the Fed decrease interest rates by 50+ bps after the April 2026 meeting?,0.05%,270972249.00%,2026-04-29


In [6]:
# use this cell to pull out a few more learnings from the markets_raw object ...
# what can you learn from this 1 data pull? 






=== Market Statistics ===
Total 24h Volume across all fetched markets: $95,049,964
Average YES probability:                     27.7%
Most liquid market:
  → Will the Fed decrease interest rates by 50+ bps after the April 2026 meeting?
  → Liquidity: $12,811,112


---
## 3️⃣ Live WebSocket Stream 🌊


> Take a look at the code below - how is this creating a simulation of a kind of "stream" from the API?Run the cell below. It will stream for ~60 seconds then stop automatically.  

> ⚠️  Press the **■ Stop** button in the toolbar to stop it early.

In [15]:
###### import websocket
import threading
import time
from IPython.display import display, clear_output

# ── Grab the top 5 token IDs to subscribe to ─────────────────────────────────
token_ids = df.nlargest(5, 'volume_24h')['clob_token_id'].tolist()
token_ids = [t for t in token_ids if t != '?'][:5]

stream_events = []
STREAM_SECONDS = 60  # auto-stop after this long

def on_open(ws):
    print(f"🟢 Connected! Subscribing to {len(token_ids)} markets...")
    sub = {"assets_ids": token_ids, "type": "Market"}
    ws.send(json.dumps(sub))

def on_message(ws, message):
    try:
        events = json.loads(message)
        if not isinstance(events, list):
            events = [events]
        for event in events:
            event_type = event.get('event_type', event.get('type', 'unknown'))
            asset_id   = event.get('asset_id', event.get('market', ''))[:12] + '...'
            price      = event.get('price', event.get('last_trade_price', None))
            stream_events.append({
                'type':     event_type,
                'asset':    asset_id,
                'price':    float(price) if price else None,
                'raw':      str(event)[:80]
            })
        clear_output(wait=True)
        sdf = pd.DataFrame(stream_events[-20:])  # show last 20 events
        print(f"🔴 LIVE STREAM — {len(stream_events)} events received so far")
        print(f"   Streaming for up to {STREAM_SECONDS}s. Stop button to end early.\n")
        display(sdf)
    except Exception as e:
        pass

def on_error(ws, error):
    print(f"⚠️ WebSocket error: {error}")

def on_close(ws, close_status_code, close_msg):
    print(f"\n🔴 Stream closed. Total events received: {len(stream_events)}")

ws = websocket.WebSocketApp(
    "wss://ws-subscriptions-clob.polymarket.com/ws/market",
    on_open=on_open,
    on_message=on_message,
    on_error=on_error,
    on_close=on_close
)

# Run in background thread, auto-close after STREAM_SECONDS
import ssl
ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE

t = threading.Thread(target=lambda: ws.run_forever(sslopt={"context": ssl_ctx}))

t.daemon = True
t.start()

time.sleep(STREAM_SECONDS)
ws.close()
print("\n✅ Stream ended.")

🔴 LIVE STREAM — 4 events received so far
   Streaming for up to 60s. Stop button to end early.



,type,asset,price,raw
0,book,500496421420...,0.30%,{'market': '0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cdc09e9d47e7d54356c6...
1,book,105509530658...,99.90%,{'market': '0x94a2b2a1b9227e44fc464614e63dfca6d4082310291c35623c3d34b70aa536...
2,price_change,0x1d2787cb8a...,NaN,{'market': '0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cdc09e9d47e7d54356c6...
3,price_change,0x1d2787cb8a...,NaN,{'market': '0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cdc09e9d47e7d54356c6...


KeyboardInterrupt: 

---
## 💬 Discussion Questions

### Datafication vs. Wisdom of Crowds - take some notes here

Start by reading the Introduction of this review on the concept of **datafication**: https://policyreview.info/concepts/datafication

1. **What is being datafied here?** What are we actually looking at when we look at this stream?

2. **Does money make the crowd wiser?** Prediction markets require skin in the game. Does that make them more accurate than polls or expert forecasts?

3. **Who is the crowd?** Look at the volume numbers. Is this really a "crowd" or a small number of large traders?

4. **Regulation:** Should governments or other regulators be able to shut down prediction markets on political events? 